In [1]:
import random

class SequenceModifier():
    ''' Modifies a sequence at a specific position'''
    def __init__(self, alphabet: list[str]):
        self.alphabet = alphabet
    
    #@with_logging(level=8)
    def _insert(self, seq: list[str], idx: int) -> None:
        insert_idx = random.choice([idx, idx + 1])
        if insert_idx <= len(seq):
            seq.insert(insert_idx, random.choice(self.alphabet))
    
    #@with_logging(level=8)
    def _replace(self, seq: list[str], idx: int) -> None:
        seq[idx] = random.choice(self.alphabet)
    
    #@with_logging(level=8)
    def _delete(self, seq: list[str], idx: int) -> None:
        if len(seq) > 1:
            seq.pop(idx)
    
    #@with_logging(level=8)
    def _swap(self, seq: list[str], idx: int) -> None:
        swap_pos = idx + random.choice([-1, 1])
        if 0 <= swap_pos < len(seq):
            seq[idx], seq[swap_pos] = seq[swap_pos], seq[idx]

In [2]:
import torch
import numpy as np
from torch.utils.data import Dataset

class CustomDataset(Dataset):
    def __init__(self, df, preprocessor, masking_weights, masking_percentage):
        """
        Args:
            df (list or array-like): The dataset of sequences.
            preprocessor: An object that can process (tokenize) text -> list of token IDs.
            masking_weights (tuple or list): The probability thresholds for the types of masking.
                Example: (p_mask, p_mask + p_random, 1.0) 
                i.e., if random < masking_weights[0], mask token
                      elif random < masking_weights[1], random token
                      else do nothing
            masking_percentage (float): Probability that a non-special token *attempts* to be replaced 
                                        by [MASK] or random. 
        """
        self.df = df
        self.preprocessor = preprocessor
        self.masking_weights = masking_weights  # e.g. (0.8, 0.9, 1.0) or something similar
        self.masking_percentage = masking_percentage
        self.ignore_index = -100

    def __len__(self):
        return len(self.df)

    def _mask(self, input_seq):
        """
        Perform in-place masking (and label creation) for MLM. Returns:
          masked_seq: The sequence with some tokens replaced by [MASK], random tokens, or left as-is.
          labels: An array with the same length as input_seq, containing the original token IDs
                  at masked positions, and -100 (self.ignore_index) at unmasked positions.
        """
        # Convert to a Python list to avoid inadvertent changes to the underlying data
        seq = input_seq.copy()

        # Keep a copy of original tokens to build the MLM labels
        labels = seq.copy()

        # Identify which tokens are maskable (i.e. not special tokens)
        special_tokens = self.preprocessor.vocab.get_special_tokens()
        mask_candidates = np.array([
            1 if token not in special_tokens else 0 
            for token in seq
        ], dtype=np.float32)

        # Probability array that determines whether each token *tries* to be masked at all
        probability_array = np.random.rand(len(seq))  # uniform(0,1)

        for idx, _ in enumerate(seq):
            # If this token is not maskable or does not pass the masking threshold -> do nothing
            if mask_candidates[idx] == 0 or probability_array[idx] >= self.masking_percentage:
                # For MLM labels, non-masked positions should be self.ignore_index
                labels[idx] = self.ignore_index
                continue

            # Otherwise, we sample from random.random() to decide how we mask
            r = random.random()

            # Example usage of masking_weights: 
            #   masking_weights = (p_mask, p_mask + p_random, 1.0)
            # if r < masking_weights[0]: --> [MASK]
            # elif r < masking_weights[1]: --> random non-special token
            # else --> do nothing
            if r < self.masking_weights[0]:
                # Replace with [MASK] token
                seq[idx] = self.preprocessor.vocab.get_id("MASK")
            elif r < self.masking_weights[1]:
                # Replace with a random non-special token
                seq[idx] = self.preprocessor.vocab.get_random_non_special_token()
            else:
                # Do nothing (i.e., keep the original token) 
                # But the label is still the original token if we are "masking" it 
                #   from a training perspective. 
                # If you want standard BERT behavior (80/10/10), 
                # in the "do nothing" branch you still keep the original, 
                # but the label is the original token. 
                pass

        return seq, labels

    def _create_attention_mask(self, input_seq):
        """
        Returns an attention mask for the input sequence: 
         - 1 where token != PAD 
         - 0 where token == PAD
        """
        pad_id = self.preprocessor.vocab.get_id("PAD")
        attention_mask = [1 if token != pad_id else 0 for token in input_seq]
        return attention_mask
    
    def _add_special_tokens(self, seq):
        """
        Adds [CLS] and [SEP] tokens to the sequence.

        Args:
            seq (list): A list of token IDs representing the sequence.

        Returns:
            list: The sequence with [CLS] and [SEP] tokens added.
        """
        cls_token_id = self.preprocessor.vocab.get_id("CLS")
        sep_token_id = self.preprocessor.vocab.get_id("SEP")

        # Add [CLS] at the start and [SEP] at the end
        return [cls_token_id] + seq + [sep_token_id]

    def __getitem__(self, idx):
        """
        Returns:
        input_ids_tensor: The masked input_ids (list of token IDs as a tensor).
        mlm_labels: The MLM labels (list of token IDs or -100 for unmasked positions as a tensor).
        attention_mask_tensor: The attention mask for the sequence as a tensor.
        """
        seq = self.df.iloc[idx]
        # Convert raw item to a list of token IDs
        preprocessed_seq = self.preprocessor.process(seq)

        # Add special tokens to finalize the sequence structure
        finalized_sequence = self._add_special_tokens(preprocessed_seq)

        # Create masked input and the MLM labels
        masked_input_ids, mlm_labels = self._mask(finalized_sequence)
        
        # Create the attention mask
        attention_mask = self._create_attention_mask(masked_input_ids)

        # Convert to torch tensors
        input_ids_tensor = torch.tensor(masked_input_ids, dtype=torch.long)
        mlm_labels_tensor = torch.tensor(mlm_labels, dtype=torch.long)
        attention_mask_tensor = torch.tensor(attention_mask, dtype=torch.long)

        return {
            "original_seq": seq,
            "input_ids": input_ids_tensor,
            "labels": mlm_labels_tensor,
            "attention_mask": attention_mask_tensor
        }

In [3]:
import torch.nn as nn
import tqdm as notebook_tqdm
from transformers import BertModel, BertConfig

class Bertax(nn.Module):
    def __init__(self, num_layers=8, 
            num_attention_heads=4, 
            hidden_size=512,
            intermediate_size=2048, 
            vocab_size=69, 
            max_position_embeddings=22, 
            num_classes=10, 
            dropout_rate=0.1):
        
        super(Bertax, self).__init__()
        
        # Initialized to pretrain mode
        self.mode = "pretrain"
        
        config = BertConfig(
            vocab_size=vocab_size,
            hidden_size=hidden_size,
            num_hidden_layers=num_layers,
            num_attention_heads=num_attention_heads,
            intermediate_size=intermediate_size,
            max_position_embeddings=max_position_embeddings,
            hidden_dropout_prob=dropout_rate,
            attention_probs_dropout_prob=dropout_rate
        )
        
        self.bert = BertModel(config)

        self.mlm_head = nn.Linear(hidden_size, vocab_size)
        self.classification_head = nn.Sequential(
            nn.Linear(hidden_size, hidden_size // 2 ),
            nn.ReLU(),
            nn.Dropout(p = dropout_rate),
            nn.Linear(hidden_size // 2, num_classes)
        )

    def preTrainMode(self):
        self.mode = "pretrain"
        
    def classifyMode(self):
        self.mode = "classify"

    def forward(self, input_ids, attention_mask = None):
        outputs = self.bert(
            input_ids = input_ids,
            attention_mask = attention_mask
            )
        sequence_output = outputs.last_hidden_state
        pooled_output = outputs.pooler_output

        if self.mode == "pretrain":
            #USE MLM-head for pre-training
            return self.mlm_head(sequence_output)
        if self.mode == "classify":
                return self.classification_head(pooled_output)
        else: 
             raise ValueError(f"Invalid mode: {self.mode}. Use 'pretrain' or 'classify'.")


/Users/filipberntsson/Documents/Studies/Thesis/Programming/BaseModel/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from datetime import datetime
import json
from tqdm import tqdm

class MLMtrainer:
    def __init__(self,
            model: nn.Module,
            train_loader: DataLoader,
            val_loader: DataLoader):
        
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.model = model.to(self.device)
        print(f"Training on device {self.device} and it is awesome!!!")
        
        self.train_loader = train_loader
        self.val_loader = val_loader
        
        self.criteronMLM = nn.CrossEntropyLoss()
        self.optimizer = optim.AdamW(self.model.parameters(), lr = 5e-6) # TODO: How to (and where) to introduce a lr-scheduler? 
        self.best_val_loss = float('inf')

    def _run_epoch(self, epoch_nr):

        self.model.train()
        total_loss, mlm_correct, total_mlm = 0, 0, 0
        progress_bar = tqdm(self.train_loader, desc=f"Training Epoch {epoch_nr + 1}", leave=False)

        ignore_index = self.train_loader.dataset.ignore_index

        for batch in progress_bar:
            input_ids = batch["input_ids"].to(self.device)
            attention_mask = batch["attention_mask"].to(self.device)  # Fixed misplaced parentheses
            mlm_labels = batch["labels"].to(self.device)

            self.optimizer.zero_grad()
            mlm_logits = self.model(input_ids, attention_mask)

            # Compute MLM loss
            loss = self.criteronMLM(
                mlm_logits.view(-1, self.model.mlm_head.out_features),
                mlm_labels.view(-1)
            )

            loss.backward()
            self.optimizer.step()

            # Accumulate total loss
            total_loss += loss.item()

            # Compute MLM accuracy
            mlm_preds = mlm_logits.argmax(dim=-1)
            mlm_correct += (mlm_preds == mlm_labels).masked_select(mlm_labels != ignore_index).sum().item()
            total_mlm += (mlm_labels != ignore_index).sum().item()

        # Calculate average loss and MLM accuracy
        avg_loss = total_loss / len(self.train_loader)
        mlm_acc = mlm_correct / total_mlm if total_mlm > 0 else 0

        return avg_loss, mlm_acc
        
    def train(self, num_epochs = 10):
        for epoch_nr in range(num_epochs):
            # Run training for one epoch
            train_avg_loss, train_mlm_acc = self._run_epoch(epoch_nr)

            # Report training metrics
            print(f"Epoch {epoch_nr + 1}/{num_epochs}")
            print(f"Train Avg Loss: {train_avg_loss:.4f}, Train MLM Accuracy: {train_mlm_acc:.4f}")



In [5]:
import torch
from preprocessing.preprocessor import Preprocessor
from preprocessing.augmentation import IdentityStrategy
from preprocessing.tokenization import KmerStrategy
from preprocessing.padding import PEndStrategy
from preprocessing.truncation import TEndStrategy

from vocab import Vocabulary, KmerVocabConstructor
from utils.dataset import fasta2pandas

import json

#Config: -------------------------------------------------------------------------------------------
FILE_PATH = 'data/raw.fasta'
SAVE_PATH = "/home/filbern/workspace/projects/testRepo/pretrained_model.pt"
NUM_EPOCHS = 1
modification_probability: float = 0.05
alphabet = ["A", "C", "G", "T"]
k = 3
optimal_length = 200 #TODO: Why, when I change this to say 200, does the last line in this cell break?
masking_limits = [0.8,0.1,0.1]
masking_percentage = 0.05

# Set up vocabulary ------------------------------------------------------------------------------

constructor = KmerVocabConstructor(k=k, alphabet=alphabet)
vocab = Vocabulary()
vocab.build_from_constructor(constructor, data=[])

vocab_path = "/home/filbern/workspace/projects/testRepo/vocab.json"
vocab.save(vocab_path)

# Set up preprocessor-------------------------------------------------------------------------------
sequence_modifier = SequenceModifier(alphabet=alphabet)

augmentation_strategy = IdentityStrategy(
    modifier = sequence_modifier,
    alphabet=alphabet,
    modification_probability = modification_probability
 )

tokenization_strategy = KmerStrategy(
    k = k,
    padding_alphabet = alphabet
    )

padding_strategy = PEndStrategy(
    optimal_length = optimal_length
)
truncation_strategy = TEndStrategy(
    optimal_length = optimal_length
)

preprocessor = Preprocessor(
        augmentation_strategy=augmentation_strategy,
        tokenization_strategy=tokenization_strategy,
        padding_strategy=padding_strategy,
        truncation_strategy=truncation_strategy,
        vocab=vocab,
    )

# Set up pre-training data ------------------------------------------------------------------------
df = fasta2pandas(FILE_PATH)
sequences = df['sequence']
dataset = CustomDataset(
    sequences,
    preprocessor,
    masking_limits,
    masking_percentage
    )

train_loader = DataLoader(
    dataset = dataset,
    batch_size = 8,
    shuffle = True
    )

val_loader = DataLoader(
    dataset = dataset,
    batch_size = 8,
    shuffle = True
    )

my_model = Bertax(
    num_layers = 4,
    hidden_size = 256,
    intermediate_size =1024,
    vocab_size = len(vocab),
    max_position_embeddings = optimal_length + 2
    )

mlm_trainer = MLMtrainer(
    my_model,
    train_loader = train_loader,
    val_loader = val_loader
    )



mlm_trainer.train(NUM_EPOCHS)
torch.save(my_model.state_dict(), SAVE_PATH)
print(f"Model's state dictionary saved to {SAVE_PATH}")

Training on device cuda and it is awesome!!!


Epoch 1/1
Train Avg Loss: 3.4794, Train MLM Accuracy: 0.2059
Model's state dictionary saved to /home/filbern/workspace/projects/pretrained_model.pt


In [ ]:
my_dict = dict


my_dict()